# Starspots: A Key to the Stellar Dynamo — Implementation\n# 항성 흑점: 항성 다이나모의 열쇠 — 구현\n\n**Paper**: Berdyugina, S. V. (2005). *Starspots: A Key to the Stellar Dynamo*. Living Reviews in Solar Physics, 2, 8.\n\n**논문**: Berdyugina, S. V. (2005). *Starspots: A Key to the Stellar Dynamo*. Living Reviews in Solar Physics, 2, 8.\n\n---\n\n## 개요 / Overview\n\n이 노트북은 Berdyugina (2005)의 리뷰 논문에서 다루는 항성 흑점 연구의 핵심 기법들을 구현합니다.\n\nThis notebook implements key techniques from Berdyugina (2005)'s review of starspot research.\n\n| Part | 주제 / Topic | 설명 / Description |\n|------|-------------|--------------------|\n| 1 | Light-Curve Inversion (LCI) | 흑점이 있는 항성의 광도 곡선 시뮬레이션 및 역산 / Simulate and invert a spotted star's light curve |\n| 2 | Doppler Imaging Principle | 회전에 의한 선 프로파일 변형 / Rotationally broadened line profile distortion |\n| 3 | Two-Band Photometry | 두 파장 측광으로 흑점 온도 결정 / Spot temperature from V and I bands |\n| 4 | Active Longitudes & Flip-Flop | 활동 경도와 flip-flop 현상 시뮬레이션 / Simulate active longitude switching |\n| 5 | Differential Rotation | 위도별 차등 회전 측정 / Measure differential rotation from spot migration |

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.optimize import minimize\nfrom matplotlib.gridspec import GridSpec\n\nplt.rcParams.update({\n    "figure.figsize": (12, 6),\n    "font.size": 12,\n    "axes.grid": True,\n    "grid.alpha": 0.3,\n})

---\n## Part 1: Light-Curve Inversion (LCI) / 광도 곡선 역산\n\n### 배경 / Background\n\n항성 흑점은 광구보다 낮은 온도를 가지며, 항성이 회전할 때 흑점이 시선 방향으로 들어오면 밝기가 감소합니다.\n이 밝기 변화(광도 곡선)로부터 흑점의 경도 분포를 역으로 추정할 수 있습니다.\n\nStarspots are cooler than the surrounding photosphere. As the star rotates, spots coming\ninto view cause brightness dips. From the observed light curve, one can invert to recover\nthe spot filling factor as a function of longitude.\n\n### 모델 / Model\n\n각 경도 구간에서의 관측 강도:\n\nObserved intensity at each rotational phase:\n\n$$I_i = f_i \\cdot I_s + (1 - f_i) \\cdot I_p$$\n\n- $f_i$: filling factor (흑점이 차지하는 면적 비율 / fractional area covered by spots)\n- $I_s = B(T_{\\text{spot}}, \\lambda)$: 흑점의 Planck 복사 / spot Planck radiance\n- $I_p = B(T_{\\text{phot}}, \\lambda)$: 광구의 Planck 복사 / photosphere Planck radiance

In [ ]:
def planck(T: float, lam: float) -> float:\n    """Compute Planck function B(T, lambda).\n\n    Args:\n        T: Temperature in Kelvin.\n        lam: Wavelength in meters.\n\n    Returns:\n        Spectral radiance in W / (m^2 sr m).\n    """\n    h = 6.626e-34  # Planck constant [J s]\n    c = 3.0e8      # Speed of light [m/s]\n    k = 1.381e-23  # Boltzmann constant [J/K]\n    return (2 * h * c**2 / lam**5) / (np.exp(h * c / (lam * k * T)) - 1)\n\n\ndef limb_darkening(mu: np.ndarray, u: float = 0.6) -> np.ndarray:\n    """Linear limb-darkening law.\n\n    Args:\n        mu: Cosine of the angle from disk center.\n        u: Limb-darkening coefficient.\n\n    Returns:\n        Limb-darkening factor.\n    """\n    return 1 - u * (1 - mu)\n\n\ndef make_spotted_star(\n    n_lon: int = 360,\n    n_lat: int = 180,\n    spots: list[dict] | None = None,\n    T_phot: float = 5800.0,\n    T_spot: float = 4200.0,\n) -> np.ndarray:\n    """Create a temperature map on a stellar sphere.\n\n    Args:\n        n_lon: Number of longitude bins.\n        n_lat: Number of latitude bins.\n        spots: List of dicts with keys 'lon', 'lat', 'radius' (all in degrees).\n        T_phot: Photosphere temperature [K].\n        T_spot: Spot temperature [K].\n\n    Returns:\n        2D temperature array of shape (n_lat, n_lon).\n    """\n    if spots is None:\n        spots = [\n            {"lon": 60, "lat": 20, "radius": 15},\n            {"lon": 200, "lat": -10, "radius": 20},\n        ]\n\n    lon = np.linspace(0, 360, n_lon, endpoint=False)\n    lat = np.linspace(-90, 90, n_lat)\n    LON, LAT = np.meshgrid(lon, lat)\n\n    temp_map = np.full_like(LON, T_phot)\n\n    for spot in spots:\n        # Angular distance from spot center on the sphere.\n        cos_dist = (\n            np.sin(np.radians(spot["lat"])) * np.sin(np.radians(LAT))\n            + np.cos(np.radians(spot["lat"]))\n            * np.cos(np.radians(LAT))\n            * np.cos(np.radians(LON - spot["lon"]))\n        )\n        ang_dist = np.degrees(np.arccos(np.clip(cos_dist, -1, 1)))\n        temp_map[ang_dist < spot["radius"]] = T_spot\n\n    return temp_map\n\n\ndef compute_light_curve(\n    temp_map: np.ndarray,\n    phases: np.ndarray,\n    inclination: float = 60.0,\n    lam: float = 550e-9,\n    u: float = 0.6,\n) -> np.ndarray:\n    """Compute the light curve of a spotted star.\n\n    Args:\n        temp_map: 2D temperature array (n_lat, n_lon).\n        phases: Rotational phases in [0, 1].\n        inclination: Stellar inclination in degrees (90 = equator-on).\n        lam: Observation wavelength [m].\n        u: Limb-darkening coefficient.\n\n    Returns:\n        Normalized flux array for each phase.\n    """\n    n_lat, n_lon = temp_map.shape\n    lon = np.linspace(0, 360, n_lon, endpoint=False)\n    lat = np.linspace(-90, 90, n_lat)\n    LON, LAT = np.meshgrid(lon, lat)\n    inc = np.radians(inclination)\n\n    # Planck radiance for each pixel.\n    B_map = planck(temp_map, lam)\n\n    # Area element: cos(lat) for spherical projection.\n    cos_lat = np.cos(np.radians(LAT))\n\n    flux = np.zeros(len(phases))\n    for i, phase in enumerate(phases):\n        sub_lon = phase * 360.0  # Sub-observer longitude.\n        # mu = cos(angle between surface normal and line of sight).\n        mu = (\n            np.sin(inc) * np.cos(np.radians(LAT)) * np.cos(np.radians(LON - sub_lon))\n            + np.cos(inc) * np.sin(np.radians(LAT))\n        )\n        visible = mu > 0\n        ld = limb_darkening(mu, u)\n        flux[i] = np.sum(B_map[visible] * mu[visible] * ld[visible] * cos_lat[visible])\n\n    # Normalize to unspotted maximum.\n    return flux / flux.max()\n\n\n# --- Simulate and plot ---\nphases = np.linspace(0, 1, 200)\ntemp_map = make_spotted_star()\nflux = compute_light_curve(temp_map, phases)\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: temperature map.\nlon = np.linspace(0, 360, temp_map.shape[1], endpoint=False)\nlat = np.linspace(-90, 90, temp_map.shape[0])\nim = axes[0].pcolormesh(lon, lat, temp_map, cmap="RdYlBu_r", shading="auto")\naxes[0].set_xlabel("Longitude [deg] / 경도 [도]")\naxes[0].set_ylabel("Latitude [deg] / 위도 [도]")\naxes[0].set_title("Stellar Temperature Map / 항성 온도 지도")\nplt.colorbar(im, ax=axes[0], label="T [K]")\n\n# Right: light curve.\naxes[1].plot(phases, flux, "k-", lw=1.5)\naxes[1].set_xlabel("Rotational Phase / 회전 위상")\naxes[1].set_ylabel("Normalized Flux / 정규화 플럭스")\naxes[1].set_title("Synthetic Light Curve / 합성 광도 곡선")\naxes[1].set_ylim(flux.min() - 0.005, 1.005)\n\nplt.tight_layout()\nplt.show()

### 광도 곡선 역산 / Light-Curve Inversion\n\n광도 곡선에서 경도별 filling factor를 복원합니다. 각 위상에서:\n\nWe recover the filling factor vs. longitude from the light curve. At each phase:\n\n$$f(\\phi) = \\frac{I_p - I_{\\text{obs}}(\\phi)}{I_p - I_s}$$\n\n여기서 $I_p$와 $I_s$는 각각 광구와 흑점의 Planck 복사입니다.\n\nwhere $I_p$ and $I_s$ are the photosphere and spot Planck radiances, respectively.

In [ ]:
# --- Light-curve inversion ---\nT_phot = 5800.0\nT_spot = 4200.0\nlam_obs = 550e-9  # V-band center [m]\n\nI_p = planck(T_phot, lam_obs)\nI_s = planck(T_spot, lam_obs)\n\n# Convert normalized flux back to absolute scale for inversion.\nflux_abs = flux * I_p  # Approximate: unspotted star emits I_p.\n\n# Inversion: recover filling factor at each phase.\nf_recovered = (I_p - flux_abs) / (I_p - I_s)\nf_recovered = np.clip(f_recovered, 0, 1)\n\n# True filling factor: compute from the temperature map.\nn_lon_bins = len(phases)\ntrue_f = np.zeros(n_lon_bins)\nfor i, phase in enumerate(phases):\n    sub_lon = phase * 360.0\n    # Sum spot pixels visible at this phase.\n    lon_grid = np.linspace(0, 360, temp_map.shape[1], endpoint=False)\n    lat_grid = np.linspace(-90, 90, temp_map.shape[0])\n    LON_g, LAT_g = np.meshgrid(lon_grid, lat_grid)\n    inc = np.radians(60.0)\n    mu = (\n        np.sin(inc) * np.cos(np.radians(LAT_g)) * np.cos(np.radians(LON_g - sub_lon))\n        + np.cos(inc) * np.sin(np.radians(LAT_g))\n    )\n    visible = mu > 0\n    spotted = temp_map < T_phot\n    cos_lat = np.cos(np.radians(LAT_g))\n    if np.sum(visible) > 0:\n        true_f[i] = np.sum((spotted & visible) * mu[visible & spotted].sum()) / (\n            np.sum(mu[visible] * cos_lat[visible])\n        ) if np.any(spotted & visible) else 0\n        # Simpler proxy: area fraction.\n        true_f[i] = np.sum(cos_lat[spotted & visible] * mu[spotted & visible]) / (\n            np.sum(cos_lat[visible] * mu[visible])\n        )\n\nfig, ax = plt.subplots(figsize=(10, 4))\nax.plot(phases * 360, true_f, "b-", lw=2, label="True filling factor / 실제 filling factor")\nax.plot(\n    phases * 360, f_recovered, "r--", lw=2,\n    label="Recovered (LCI) / 복원된 filling factor",\n)\nax.set_xlabel("Longitude [deg] / 경도 [도]")\nax.set_ylabel("Filling Factor $f$")\nax.set_title("Light-Curve Inversion Result / 광도 곡선 역산 결과")\nax.legend()\nax.set_xlim(0, 360)\nplt.tight_layout()\nplt.show()

---\n## Part 2: Doppler Imaging Principle / Doppler Imaging 원리\n\n### 배경 / Background\n\n빠르게 회전하는 항성에서 흡수선은 회전에 의해 broadening됩니다.\n흑점은 국소적으로 연속 복사를 줄이므로 흡수선 프로파일에 "bump"(돌기)를 만듭니다.\n항성이 회전하면 bump가 프로파일의 blue 쪽에서 red 쪽으로 이동합니다.\n\nIn a rapidly rotating star, absorption lines are rotationally broadened.\nA cool spot reduces the local continuum, creating a "bump" in the absorption line profile.\nAs the star rotates, the bump migrates from the blue wing to the red wing.\n\n### 핵심 개념 / Key Concept\n\n- $v \\sin i = 50$ km/s: 시선 방향 투영 회전 속도 / projected rotational velocity\n- 흑점 위치의 Doppler shift: $\\Delta v = v \\sin i \\cdot \\cos(\\text{lat}) \\cdot \\sin(\\text{lon} - \\phi)$\n- bump의 위치가 경도를, 크기가 면적을, 이동 패턴이 위도를 알려줍니다.\n- The bump position encodes longitude, its amplitude encodes area, and its migration pattern encodes latitude.

In [ ]:
def rotationally_broadened_profile(\n    velocity: np.ndarray,\n    vsini: float = 50.0,\n    u: float = 0.6,\n) -> np.ndarray:\n    """Compute a rotationally broadened absorption line profile.\n\n    Uses the analytic Gray (2005) kernel for linear limb darkening.\n\n    Args:\n        velocity: Velocity grid [km/s].\n        vsini: Projected rotational velocity [km/s].\n        u: Limb-darkening coefficient.\n\n    Returns:\n        Normalized rotational broadening kernel.\n    """\n    x = velocity / vsini\n    G = np.zeros_like(x)\n    mask = np.abs(x) < 1.0\n    x_m = x[mask]\n    c1 = 2 * (1 - u) / (np.pi * vsini * (1 - u / 3))\n    c2 = u / (2 * vsini * (1 - u / 3))\n    G[mask] = c1 * np.sqrt(1 - x_m**2) + c2 * (1 - x_m**2)\n    return G\n\n\ndef spotted_line_profile(\n    velocity: np.ndarray,\n    vsini: float,\n    spot_lon: float,\n    spot_lat: float,\n    spot_radius: float,\n    phase: float,\n    inclination: float = 60.0,\n    u: float = 0.6,\n    intrinsic_width: float = 5.0,\n    line_depth: float = 0.7,\n    spot_contrast: float = 0.3,\n    n_grid: int = 100,\n) -> np.ndarray:\n    """Compute absorption line profile with a spot bump.\n\n    Args:\n        velocity: Velocity grid [km/s].\n        vsini: Projected rotational velocity [km/s].\n        spot_lon: Spot longitude [deg].\n        spot_lat: Spot latitude [deg].\n        spot_radius: Spot angular radius [deg].\n        phase: Rotational phase [0, 1].\n        inclination: Inclination [deg].\n        u: Limb-darkening coefficient.\n        intrinsic_width: Intrinsic Gaussian line width [km/s].\n        line_depth: Central depth of the absorption line.\n        spot_contrast: Spot continuum reduction factor (0 = dark, 1 = same as photosphere).\n        n_grid: Grid resolution per hemisphere.\n\n    Returns:\n        Normalized flux profile.\n    """\n    inc = np.radians(inclination)\n    sub_lon = phase * 360.0\n\n    # Build grid on visible hemisphere.\n    lat_arr = np.linspace(-90, 90, n_grid)\n    lon_arr = np.linspace(sub_lon - 90, sub_lon + 90, n_grid)\n    LON, LAT = np.meshgrid(lon_arr, lat_arr)\n\n    # mu (visibility).\n    mu = (\n        np.sin(inc) * np.cos(np.radians(LAT)) * np.cos(np.radians(LON - sub_lon))\n        + np.cos(inc) * np.sin(np.radians(LAT))\n    )\n    visible = mu > 0\n\n    # Doppler shift of each surface element.\n    v_shift = vsini * np.cos(np.radians(LAT)) * np.sin(np.radians(LON - sub_lon))\n\n    # Identify spot pixels.\n    cos_dist = (\n        np.sin(np.radians(spot_lat)) * np.sin(np.radians(LAT))\n        + np.cos(np.radians(spot_lat))\n        * np.cos(np.radians(LAT))\n        * np.cos(np.radians(LON - spot_lon))\n    )\n    ang_dist = np.degrees(np.arccos(np.clip(cos_dist, -1, 1)))\n    is_spot = ang_dist < spot_radius\n\n    # Continuum weight: photosphere = 1, spot = spot_contrast.\n    weight = np.where(is_spot, spot_contrast, 1.0)\n    ld = limb_darkening(mu, u)\n\n    # Build profile by summing shifted intrinsic lines.\n    profile = np.zeros_like(velocity)\n    continuum = 0.0\n    for i in range(n_grid):\n        for j in range(n_grid):\n            if not visible[i, j]:\n                continue\n            w = weight[i, j] * mu[i, j] * ld[i, j] * np.cos(np.radians(LAT[i, j]))\n            # Gaussian intrinsic line.\n            local_line = line_depth * np.exp(\n                -0.5 * ((velocity - v_shift[i, j]) / intrinsic_width) ** 2\n            )\n            profile += w * local_line\n            continuum += w\n\n    if continuum > 0:\n        profile = 1.0 - profile / continuum\n\n    return profile\n\n\n# --- Compute line profiles at several phases ---\nvsini = 50.0  # km/s\nvelocity = np.linspace(-80, 80, 300)\nspot_lon = 60.0\nspot_lat = 20.0\nspot_radius = 15.0\n\nphases_di = np.linspace(0.0, 0.8, 9)\n\nfig, ax = plt.subplots(figsize=(10, 8))\n\nfor k, ph in enumerate(phases_di):\n    profile = spotted_line_profile(\n        velocity, vsini, spot_lon, spot_lat, spot_radius, ph,\n        n_grid=60,\n    )\n    # Also compute unspotted profile for reference.\n    unspotted = spotted_line_profile(\n        velocity, vsini, spot_lon, spot_lat, 0.0, ph,\n        n_grid=60,\n    )\n    offset = k * 0.15\n    ax.plot(velocity, profile + offset, "b-", lw=1.2)\n    ax.plot(velocity, unspotted + offset, "k--", lw=0.5, alpha=0.5)\n    ax.text(75, 1.0 + offset - 0.02, f"$\\phi$={ph:.2f}", fontsize=9)\n\nax.set_xlabel("Velocity [km/s] / 속도 [km/s]")\nax.set_ylabel("Flux + offset / 플럭스 + 오프셋")\nax.set_title(\n    "Doppler Imaging: Spot Bump Migration / Doppler Imaging: 흑점 bump 이동\\n"\n    f"(spot at lon={spot_lon}°, lat={spot_lat}°, $v\\\\sin i$={vsini} km/s)"\n)\nax.axvline(0, color="gray", ls=":", alpha=0.5)\nax.axvline(-vsini, color="gray", ls=":", alpha=0.3)\nax.axvline(vsini, color="gray", ls=":", alpha=0.3)\nplt.tight_layout()\nplt.show()

---\n## Part 3: Starspot Temperature from Two-Band Photometry / 두 파장 측광으로 흑점 온도 결정\n\n### 배경 / Background\n\n단일 파장 광도 곡선에서는 흑점의 온도($T_s$)와 면적(filling factor $f$)이 축퇴(degenerate)합니다.\n두 개의 파장(예: V, I band)을 동시에 관측하면 이 축퇴를 깰 수 있습니다.\n\nWith a single-band light curve, spot temperature and filling factor are degenerate.\nSimultaneous observations in two bands (e.g., V and I) break this degeneracy.\n\n### 모델 / Model\n\n각 파장에서의 등급 변화:\n\nMagnitude change in each band:\n\n$$\\Delta m_\\lambda = -2.5 \\log_{10}\\left[1 - f\\left(1 - \\frac{B(T_s, \\lambda)}{B(T_p, \\lambda)}\\right)\\right]$$\n\n색지수 변화:\n\nColor change:\n\n$$\\Delta(V-I) = \\Delta m_V - \\Delta m_I$$\n\n흑점이 보일 때 $V-I$가 커집니다 (더 붉어짐).\n\nWhen the spot faces the observer, $V-I$ increases (becomes redder).

In [ ]:
def delta_mag(f: float, T_spot: float, T_phot: float, lam: float) -> float:
    """Compute magnitude change due to a spot.

    Args:
        f: Filling factor.
        T_spot: Spot temperature [K].
        T_phot: Photosphere temperature [K].
        lam: Wavelength [m].

    Returns:
        Magnitude change (positive = fainter).
    """
    ratio = planck(T_spot, lam) / planck(T_phot, lam)
    return -2.5 * np.log10(1 - f * (1 - ratio))


# --- Parameter space exploration ---
T_phot = 5800.0
lam_V = 550e-9   # V-band
lam_I = 800e-9   # I-band

# Grid of (Delta_T, f).
filling_factors = np.linspace(0.01, 0.40, 200)
delta_T_values = np.linspace(200, 2500, 200)  # T_phot - T_spot

F, DT = np.meshgrid(filling_factors, delta_T_values)
T_spots = T_phot - DT

# Compute delta_m for V and I.
dm_V = np.vectorize(delta_mag)(F, T_spots, T_phot, lam_V)
dm_I = np.vectorize(delta_mag)(F, T_spots, T_phot, lam_I)
delta_color = dm_V - dm_I  # Delta(V-I)

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# V-band magnitude change.
c0 = axes[0].contourf(F, DT, dm_V, levels=20, cmap="YlOrRd")
axes[0].set_xlabel("Filling Factor $f$")
axes[0].set_ylabel(r"$\Delta T = T_p - T_s$ [K]")
axes[0].set_title(r"$\Delta m_V$ / V 등급 변화")
plt.colorbar(c0, ax=axes[0], label=r"$\Delta m_V$ [mag]")

# I-band magnitude change.
c1 = axes[1].contourf(F, DT, dm_I, levels=20, cmap="YlOrRd")
axes[1].set_xlabel("Filling Factor $f$")
axes[1].set_ylabel(r"$\Delta T = T_p - T_s$ [K]")
axes[1].set_title(r"$\Delta m_I$ / I 등급 변화")
plt.colorbar(c1, ax=axes[1], label=r"$\Delta m_I$ [mag]")

# Color change.
c2 = axes[2].contourf(F, DT, delta_color, levels=20, cmap="RdBu_r")
axes[2].set_xlabel("Filling Factor $f$")
axes[2].set_ylabel(r"$\Delta T = T_p - T_s$ [K]")
axes[2].set_title(r"$\Delta(V-I)$ / 색지수 변화")
plt.colorbar(c2, ax=axes[2], label=r"$\Delta(V-I)$ [mag]")

# Mark an example observation: Delta_mV = 0.1, Delta_color = 0.03.
for ax in axes:
    ax.contour(F, DT, dm_V, levels=[0.10], colors="cyan", linewidths=2, linestyles="--")
    ax.contour(F, DT, delta_color, levels=[0.03], colors="lime", linewidths=2, linestyles="-")

plt.tight_layout()
plt.show()

print("Cyan dashed = delta_mV = 0.10 mag constraint / delta_mV = 0.10 등급 제한")
print("Green solid = delta(V-I) = 0.03 mag constraint / delta(V-I) = 0.03 색지수 제한")
print("Intersection gives unique (delta_T, f) solution / 교차점이 유일한 해를 줌")

---
## Part 4: Active Longitudes and Flip-Flop / 활동 경도와 Flip-Flop 현상

### 배경 / Background

많은 활동 항성에서 흑점은 ~180도 떨어진 두 개의 "활동 경도"(active longitudes)에 집중됩니다.
한 경도가 수 회전 주기 동안 더 활발하다가, 갑자기 반대편 경도로 활동이 "전환"(flip-flop)됩니다.
이 현상은 비축대칭(non-axisymmetric) 다이나모 모드의 증거입니다.

Many active stars show spots concentrated at two "active longitudes" ~180 degrees apart.
One longitude dominates for several rotation periods, then activity abruptly "flips" to
the opposite longitude. This flip-flop phenomenon is evidence for non-axisymmetric dynamo modes.

### 모델 / Model

두 활동 경도 $\lambda_1$과 $\lambda_2 = \lambda_1 + 180°$에서 흑점 활동 강도를 시간에 따라 시뮬레이션합니다.
Flip-flop 주기 $P_{\text{ff}}$마다 활동이 전환됩니다.

We simulate spot activity at two active longitudes $\lambda_1$ and $\lambda_2 = \lambda_1 + 180°$,
with activity switching every flip-flop period $P_{\text{ff}}$.

In [ ]:
def simulate_flip_flop(
    n_rotations: int = 60,
    P_rot: float = 3.0,
    P_flipflop: float = 30.0,
    active_lon_1: float = 90.0,
    spot_scatter: float = 15.0,
    base_activity: float = 0.1,
    active_activity: float = 0.8,
    rng_seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate flip-flop activity between two active longitudes.

    Args:
        n_rotations: Number of stellar rotations to simulate.
        P_rot: Rotation period [days].
        P_flipflop: Flip-flop period [days].
        active_lon_1: First active longitude [deg].
        spot_scatter: Gaussian scatter around active longitude [deg].
        base_activity: Background spot filling factor.
        active_activity: Active longitude spot filling factor.
        rng_seed: Random seed for reproducibility.

    Returns:
        Tuple of (times [days], longitudes [deg], filling_factors).
    """
    rng = np.random.default_rng(rng_seed)
    active_lon_2 = (active_lon_1 + 180.0) % 360.0

    times = []
    lons = []
    fills = []

    total_time = n_rotations * P_rot
    n_spots_per_rot = 8  # Spots emerging per rotation.

    for rot in range(n_rotations):
        t_base = rot * P_rot

        # Determine which longitude is active.
        cycle_phase = (t_base % (2 * P_flipflop)) / P_flipflop
        if cycle_phase < 1.0:
            primary_lon = active_lon_1
            secondary_lon = active_lon_2
        else:
            primary_lon = active_lon_2
            secondary_lon = active_lon_1

        for _ in range(n_spots_per_rot):
            t = t_base + rng.uniform(0, P_rot)

            # 70% chance at primary, 20% at secondary, 10% random.
            r = rng.random()
            if r < 0.70:
                lon = rng.normal(primary_lon, spot_scatter) % 360
                f = rng.uniform(0.5, 1.0) * active_activity
            elif r < 0.90:
                lon = rng.normal(secondary_lon, spot_scatter) % 360
                f = rng.uniform(0.2, 0.5) * active_activity
            else:
                lon = rng.uniform(0, 360)
                f = rng.uniform(0.05, 0.3) * base_activity

            times.append(t)
            lons.append(lon)
            fills.append(f)

    return np.array(times), np.array(lons), np.array(fills)


# --- Simulate and plot ---
times, lons, fills = simulate_flip_flop(
    n_rotations=80,
    P_rot=3.0,
    P_flipflop=40.0,
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: longitude-time diagram.
sc = axes[0].scatter(times, lons, c=fills, s=fills * 80, cmap="hot_r",
                     alpha=0.7, edgecolors="none", vmin=0, vmax=0.8)
axes[0].axhline(90, color="cyan", ls="--", alpha=0.5, label="Active lon 1 (90 deg)")
axes[0].axhline(270, color="lime", ls="--", alpha=0.5, label="Active lon 2 (270 deg)")
axes[0].set_ylabel("Longitude [deg] / 경도 [도]")
axes[0].set_title("Active Longitude Diagram / 활동 경도 다이어그램")
axes[0].set_ylim(0, 360)
axes[0].legend(loc="upper right")
plt.colorbar(sc, ax=axes[0], label="Filling factor")

# Mark flip-flop transitions.
P_ff = 40.0
for t_flip in np.arange(P_ff, times.max(), P_ff):
    axes[0].axvline(t_flip, color="red", ls=":", alpha=0.4)

# Bottom: activity at each longitude over time (binned).
n_time_bins = 40
time_edges = np.linspace(0, times.max(), n_time_bins + 1)
activity_1 = np.zeros(n_time_bins)
activity_2 = np.zeros(n_time_bins)

for k in range(n_time_bins):
    mask = (times >= time_edges[k]) & (times < time_edges[k + 1])
    near_1 = np.abs(((lons[mask] - 90 + 180) % 360) - 180) < 45
    near_2 = np.abs(((lons[mask] - 270 + 180) % 360) - 180) < 45
    activity_1[k] = np.sum(fills[mask][near_1]) if np.any(near_1) else 0
    activity_2[k] = np.sum(fills[mask][near_2]) if np.any(near_2) else 0

time_centers = 0.5 * (time_edges[:-1] + time_edges[1:])
axes[1].plot(time_centers, activity_1, "c-o", lw=2, ms=4, label="Lon 1 (90 deg)")
axes[1].plot(time_centers, activity_2, "g-s", lw=2, ms=4, label="Lon 2 (270 deg)")
axes[1].set_xlabel("Time [days] / 시간 [일]")
axes[1].set_ylabel("Total Activity / 총 활동도")
axes[1].set_title("Flip-Flop Pattern / Flip-Flop 패턴")
axes[1].legend()

for t_flip in np.arange(P_ff, times.max(), P_ff):
    axes[1].axvline(t_flip, color="red", ls=":", alpha=0.4, label="" if t_flip > P_ff else "Flip-flop")

plt.tight_layout()
plt.show()

print(f"Flip-flop period / Flip-flop 주기: P_ff = {P_ff} days")
print(f"Rotation period / 회전 주기: P_rot = 3.0 days")
print(f"Number of rotations per flip-flop / Flip-flop당 회전 수: {P_ff / 3.0:.0f}")

---
## Part 5: Differential Rotation from Spot Migration / 흑점 이동으로부터 차등 회전 측정

### 배경 / Background

항성의 차등 회전(differential rotation)은 위도에 따라 회전 속도가 다른 현상입니다.
서로 다른 위도에 있는 흑점들의 경도 이동률(drift rate)을 비교하면
차등 회전 파라미터를 측정할 수 있습니다.

Differential rotation means the stellar rotation rate varies with latitude.
By comparing the longitude drift rates of spots at different latitudes,
one can measure the differential rotation parameter.

### 모델 / Model

각속도 법칙 / Angular velocity law:

$$\Omega(\theta) = \Omega_{\text{eq}} - \Delta\Omega \sin^2\theta$$

여기서 / where:
- $\theta$: 위도 / latitude
- $\Omega_{\text{eq}}$: 적도 각속도 / equatorial angular velocity
- $\Delta\Omega$: 차등 회전 크기 / differential rotation amplitude
- $k = \Delta\Omega / \Omega_{\text{eq}}$: 상대적 차등 회전 파라미터 / relative differential rotation parameter

태양의 경우 $k \approx 0.19$이며, 빠르게 회전하는 항성은 $k$가 더 작은 경향이 있습니다.

For the Sun, $k \approx 0.19$; rapidly rotating stars tend to have smaller $k$.

In [ ]:
def differential_rotation(
    lat_deg: float,
    Omega_eq: float = 360.0 / 25.0,
    delta_Omega: float = 360.0 / 25.0 * 0.19,
) -> float:
    """Compute angular velocity at a given latitude.

    Args:
        lat_deg: Latitude in degrees.
        Omega_eq: Equatorial angular velocity [deg/day].
        delta_Omega: Differential rotation amplitude [deg/day].

    Returns:
        Angular velocity [deg/day].
    """
    return Omega_eq - delta_Omega * np.sin(np.radians(lat_deg)) ** 2


def simulate_spot_migration(
    latitudes: list[float],
    n_days: int = 100,
    Omega_eq: float = 14.4,
    k: float = 0.19,
    noise_std: float = 1.0,
    rng_seed: int = 42,
) -> dict:
    """Simulate spot longitude drift at different latitudes.

    Args:
        latitudes: List of spot latitudes [deg].
        n_days: Number of days to simulate.
        Omega_eq: Equatorial angular velocity [deg/day].
        k: Relative differential rotation parameter.
        noise_std: Observational noise in longitude [deg].
        rng_seed: Random seed.

    Returns:
        Dict mapping latitude to (times, longitudes) arrays.
    """
    rng = np.random.default_rng(rng_seed)
    delta_Omega = k * Omega_eq
    results = {}

    for lat in latitudes:
        omega = differential_rotation(lat, Omega_eq, delta_Omega)
        times = np.arange(0, n_days, 1.0)
        # Longitude in co-rotating equatorial frame: drift = (omega - Omega_eq) * t.
        lon_drift = (omega - Omega_eq) * times
        lon_observed = lon_drift + rng.normal(0, noise_std, len(times))
        results[lat] = (times, lon_observed)

    return results


# --- Simulate spot migration ---
latitudes = [0, 10, 20, 30, 45, 60]
Omega_eq = 14.4  # deg/day (solar-like)
k_true = 0.19    # Solar value.

spot_data = simulate_spot_migration(latitudes, n_days=100, Omega_eq=Omega_eq, k=k_true)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: longitude drift over time.
colors = plt.cm.viridis(np.linspace(0, 0.9, len(latitudes)))
measured_drifts = {}

for (lat, (t, lon)), color in zip(spot_data.items(), colors):
    axes[0].plot(t, lon, ".", color=color, ms=2, alpha=0.5)
    # Linear fit to get drift rate.
    coeffs = np.polyfit(t, lon, 1)
    drift_rate = coeffs[0]
    measured_drifts[lat] = drift_rate
    axes[0].plot(t, np.polyval(coeffs, t), "-", color=color, lw=2,
                 label=f"lat={lat}: {drift_rate:.3f} deg/day")

axes[0].set_xlabel("Time [days] / 시간 [일]")
axes[0].set_ylabel("Longitude Drift [deg] / 경도 이동 [도]")
axes[0].set_title("Spot Migration / 흑점 이동")
axes[0].legend(fontsize=8)

# Right: recover differential rotation law.
lats_fit = np.array(list(measured_drifts.keys()))
drifts_fit = np.array(list(measured_drifts.values()))

# The drift rate relative to equator = -delta_Omega * sin^2(lat).
sin2_lat = np.sin(np.radians(lats_fit)) ** 2

# Fit: drift = -delta_Omega * sin^2(lat).
from scipy.optimize import curve_fit

def dr_model(sin2, delta_omega):
    """Linear model for drift rate vs sin^2(lat)."""
    return -delta_omega * sin2

popt, pcov = curve_fit(dr_model, sin2_lat, drifts_fit)
delta_Omega_recovered = popt[0]
k_recovered = delta_Omega_recovered / Omega_eq

lat_smooth = np.linspace(0, 70, 100)
sin2_smooth = np.sin(np.radians(lat_smooth)) ** 2

axes[1].plot(lats_fit, drifts_fit, "ko", ms=8, label="Measured / 측정값")
axes[1].plot(lat_smooth, dr_model(sin2_smooth, delta_Omega_recovered), "r-", lw=2,
             label=f"Fit: k = {k_recovered:.3f}")
axes[1].plot(lat_smooth, dr_model(sin2_smooth, k_true * Omega_eq), "b--", lw=1.5,
             label=f"True: k = {k_true:.3f}")
axes[1].set_xlabel("Latitude [deg] / 위도 [도]")
axes[1].set_ylabel("Drift Rate [deg/day] / 이동률 [도/일]")
axes[1].set_title("Recovered Differential Rotation / 복원된 차등 회전")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"True k = {k_true:.3f}")
print(f"Recovered k = {k_recovered:.3f}")
print(f"True delta_Omega = {k_true * Omega_eq:.3f} deg/day")
print(f"Recovered delta_Omega = {delta_Omega_recovered:.3f} deg/day")
print()
print("Comparison with other stars / 다른 항성과의 비교:")
print(f"  Sun (solar): k ~ 0.19")
print(f"  AB Dor (rapid rotator): k ~ 0.004")
print(f"  HR 1099 (RS CVn): k ~ 0.002")
print(f"  LQ Hya (young active): k ~ 0.01")

---
## Summary / 요약

이 노트북에서 구현한 내용과 Berdyugina (2005) 논문의 해당 섹션을 정리합니다.

This table maps each notebook part to the corresponding section in Berdyugina (2005).

| Part | 구현 내용 / Implementation | 논문 섹션 / Paper Section | 핵심 결과 / Key Result |
|------|--------------------------|--------------------------|----------------------|
| 1 | Light-Curve Inversion (LCI) | Sec. 4.1 — Photometry, Light-curve modeling | 광도 곡선에서 filling factor 복원 가능하나 위도 정보 부족 / Filling factor recoverable from light curve but latitude information is lost |
| 2 | Doppler Imaging Principle | Sec. 4.2 — Spectroscopy, Doppler imaging | 흑점이 회전 broadened 선 프로파일에 bump를 생성; 위치가 경도/위도를 인코딩 / Spots create bumps in rotationally broadened profiles; position encodes longitude/latitude |
| 3 | Two-Band Photometry | Sec. 5.1 — Starspot temperatures | 두 파장 관측으로 온도-면적 축퇴를 깰 수 있음 / Two-band observations break the temperature-area degeneracy |
| 4 | Active Longitudes & Flip-Flop | Sec. 6.1 — Active longitudes, Sec. 6.2 — Flip-flop phenomenon | 비축대칭 다이나모 모드의 증거; 주기적 활동 전환 / Evidence for non-axisymmetric dynamo modes; periodic activity switching |
| 5 | Differential Rotation | Sec. 7 — Differential rotation | 흑점 이동에서 차등 회전 측정; 빠른 회전 항성일수록 작은 k값 / Differential rotation measured from spot migration; faster rotators have smaller k |

### 참고문헌 / References

- Berdyugina, S. V. (2005). "Starspots: A Key to the Stellar Dynamo". *Living Reviews in Solar Physics*, 2, 8. [DOI: 10.12942/lrsp-2005-8](https://doi.org/10.12942/lrsp-2005-8)
- Gray, D. F. (2005). *The Observation and Analysis of Stellar Photospheres*. Cambridge University Press.
- Vogt, S. S. & Penrod, G. D. (1983). "Doppler Imaging of spotted stars". *PASP*, 95, 565.